In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install camel-tools
!pip install -q urlextract
!pip install PyArabic

In [ ]:
!pip install -q google-genai

In [ ]:
import re
import pandas as pd

import pandas as pd
import google.generativeai as genai
from google.genai.types import HarmCategory, HarmBlockThreshold
import time

from math import ceil


In [ ]:
!git clone https://github.com/iabufarha/ArSarcasm-v2.git

s=pd.read_csv('ArSarcasm-v2/ArSarcasm-v2/training_data.csv')['tweet'].str.split().str.len()

smallest_num_words = s.min()
largest_num_words = s.max()

print(f"Smallest number of words in a tweet: {smallest_num_words}")
print(f"Largest number of words in a tweet: {largest_num_words}")

Cloning into 'ArSarcasm-v2'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 33 (delta 15), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 1.04 MiB | 3.72 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Smallest number of words in a tweet: 1
Largest number of words in a tweet: 61


In [ ]:
import pandas as pd
import re

from camel_tools.utils.normalize import normalize_unicode
from camel_tools.utils.normalize import normalize_alef_maksura_ar
from camel_tools.utils.normalize import normalize_alef_ar
from camel_tools.utils.normalize import normalize_teh_marbuta_ar
from camel_tools.utils.dediac import dediac_ar

from urlextract import URLExtract

from pyarabic.araby import strip_tatweel

In [ ]:
arabic_to_latin_numbers = { '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9' }

In [ ]:
extractor = URLExtract()

def clean_text(text):
  text = normalize_unicode(text) # ﷺ -> صلى الله عليه وسلم
  text = normalize_alef_ar(text) # Normalize alef variants to 'ا'
  text = normalize_alef_maksura_ar(text) # Normalize alef maksura 'ى' to yeh 'ي'
  text = normalize_teh_marbuta_ar(text) # Normalize teh marbuta 'ة' to heh 'ه'
  text = dediac_ar(text) # Dediacritization
  text = text.translate(str.maketrans(arabic_to_latin_numbers))

  urls = extractor.find_urls(text)
  for url in urls:
      text = text.replace(url, '')

  text = strip_tatweel(text)    # e.g. الســـــــــــــــــــعودية -> السعودية

  text = re.sub(r'(.)\1{2,}', r'\1\1', text)  # Replace 3+ consecutive chars with just 2

  # text = re.sub(r'[^ء-يa-zA-Z0-9\s\(\)\[\]\{\}]+', ' ', text) # Remove all characters except Arabic letters, English letters, digits, # whitespace, and specific punctuation marks (parentheses, brackets, braces).
  text = re.sub(r'[^ء-ي\s]+', ' ', text)
  text = re.sub(r'\s+', ' ', text).strip() # Collapse multiple spaces into a single space and trim leading/trailing spaces.

  return text

In [ ]:
import random

def generate_random_numbers(count):
  return [random.randint(smallest_num_words, largest_num_words) for _ in range(count)]

In [ ]:
SELECTED_MODEL = 'gemini-2.5-pro-preview-03-25'
max_output_tokens =  65536

batch_size = 30

In [ ]:
prompt_configs = [
    {"sarcasm": False, "sentiment": "NEU", "dialect": "egypt", "number": 335},
    {"sarcasm": False, "sentiment": "NEG", "dialect": "egypt", "number": 514},
    {"sarcasm": False, "sentiment": "POS", "dialect": "egypt", "number": 536},
    {"sarcasm": False, "sentiment": "NEU", "dialect": "levant", "number": 828},
    {"sarcasm": False, "sentiment": "NEU", "dialect": "gulf", "number": 835},
    {"sarcasm": False, "sentiment": "NEG", "dialect": "levant", "number": 848},
    {"sarcasm": False, "sentiment": "POS", "dialect": "levant", "number": 869},
    {"sarcasm": False, "sentiment": "POS", "dialect": "gulf", "number": 938},
    {"sarcasm": False, "sentiment": "NEG", "dialect": "gulf", "number": 947},
    {"sarcasm": False, "sentiment": "NEG", "dialect": "magreb", "number": 989},
    {"sarcasm": False, "sentiment": "NEU", "dialect": "magreb", "number": 990},
    {"sarcasm": False, "sentiment": "POS", "dialect": "magreb", "number": 994},

    {"sarcasm": True, "sentiment": "NEG", "dialect": "egypt", "number": 760},
    {"sarcasm": True, "sentiment": "NEG", "dialect": "msa", "number": 1074},
    {"sarcasm": True, "sentiment": "NEG", "dialect": "levant", "number": 1388},
    {"sarcasm": True, "sentiment": "NEG", "dialect": "gulf", "number": 1404},
    {"sarcasm": True, "sentiment": "NEU", "dialect": "egypt", "number": 1456},
    {"sarcasm": True, "sentiment": "NEU", "dialect": "msa", "number": 1470},
    {"sarcasm": True, "sentiment": "NEG", "dialect": "magreb", "number": 1487},
    {"sarcasm": True, "sentiment": "NEU", "dialect": "levant", "number": 1492},
    {"sarcasm": True, "sentiment": "NEU", "dialect": "gulf", "number": 1493},
    {"sarcasm": True, "sentiment": "NEU", "dialect": "magreb", "number": 1499}
]

In [ ]:
def create_model():
    with open('/content/drive/MyDrive/Project/secrets/Gemini-API-Key.txt', 'r') as f:
        GEMINI_API_KEY = f.read().strip()

    safety_settings_options = {
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    }

    genai.configure(api_key=GEMINI_API_KEY)

    generation_config = {
        "temperature": 1,  # Controls randomness (0 = deterministic, 1 = highly random)
        "top_p": 0.95,     # Higher values make the model more creative
        "max_output_tokens": max_output_tokens,
        "response_mime_type": "text/plain",
    }

    model = genai.GenerativeModel(
        model_name=SELECTED_MODEL,
        generation_config=generation_config,
        safety_settings=safety_settings_options,
    )
    print(model)
    return model

model =     create_model()

genai.GenerativeModel(
    model_name='models/gemini-2.5-pro-preview-03-25',
    generation_config={'temperature': 1, 'top_p': 0.95, 'max_output_tokens': 65536, 'response_mime_type': 'text/plain'},
    safety_settings={<HarmCategory.HARM_CATEGORY_HARASSMENT: 7>: <HarmBlockThreshold.BLOCK_NONE: 4>, <HarmCategory.HARM_CATEGORY_HATE_SPEECH: 8>: <HarmBlockThreshold.BLOCK_NONE: 4>, <HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 9>: <HarmBlockThreshold.BLOCK_NONE: 4>, <HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 10>: <HarmBlockThreshold.BLOCK_NONE: 4>},
    tools=None,
    system_instruction=None,
    cached_content=None
)


In [ ]:
def create_prompt(sarcasm, sentiment, dialect, remaining_count=None):
    # Map sentiment codes to full words
    sentiment_map = {
        "NEG": "negative",
        "NEU": "neutral",
        "POS": "positive"
    }

    # Map dialect codes to full words
    dialect_map = {
        "egypt": "egyptian",
        "gulf": "gulf",
        "levant": "levantine",
        "magreb": "magrebi",
        "msa": "Modern Standard Arabic"
    }

    # If no remaining_count is provided, use the default batch_size
    count = remaining_count if remaining_count is not None else batch_size

    sarcastic_prompt = ''

    if sarcasm:
      sarcastic_prompt = 'sarcastic tweets'

    return f"Generate {count} Arabic {sarcastic_prompt} tweets that have {sentiment_map[sentiment]} sentiment and {dialect_map[dialect]} dialect. Each tweet must be in one exact line. Give no other strings after or before the tweets. Give no explanation. Number of lines must exactly be {count}. Number of words per tweets must be: {generate_random_numbers(count)}. Tweets should have different contexts and must not be repeated."


In [ ]:
def print_df_info(name, df):
  print('*'*5)
  acopy = df.copy()
  print(f'len({name}) = {len(df)}')
  print('')
  print(acopy.groupby(['sarcasm']).size().reset_index(name='count'))
  print('')
  print(acopy.groupby(['sentiment']).size().reset_index(name='count'))
  print(acopy.groupby(['dialect']).size().reset_index(name='count'))
  print('')
  print(acopy.groupby('combined_label').size().reset_index(name='count').sort_values(by='count'))
  print('')
  print('*'*5)

In [ ]:
def generate_tweet_batch(prompt, chat_gen):
    try:
        response = None
        try:
            response = chat_gen.send_message(prompt).text
        except Exception as e:
            print(f"API call failed : {str(e)}")
            return []

        # Clean up the response
        tweets = re.sub(r'\n\s*\n', '\n', response)  # Remove empty lines
        tweets = tweets.split('\n')

        # Filter out empty tweets and clean them
        tweets = [tweet.strip() for tweet in tweets if tweet.strip()]

        return tweets

    except Exception as e:
        print(f"Error in tweet generation: {str(e)}")
        # Return empty list in case of error
        return []


In [ ]:
def generate_numbered_prompts():
    for config in prompt_configs:
        prompt = create_prompt(config["sarcasm"], config["sentiment"], config["dialect"])
        print(f"{prompt}    {config['number']}")

In [ ]:
def generate_and_save_tweets():
    # Create a list to store all generated tweets with their metadata
    all_tweets = []
    chat = model.start_chat()

    for config in prompt_configs:
        sarcasm = config["sarcasm"]
        sentiment = config["sentiment"]
        dialect = config["dialect"]
        combined_label = f"{sarcasm},{sentiment},{dialect}"
        target_number = config["number"]

        # Track how many valid tweets we've collected for this configuration
        collected_tweets = 0

        # Keep generating batches until we reach the target number
        while collected_tweets < target_number:
            # Calculate how many more tweets we need
            remaining = target_number - collected_tweets

            # Determine batch size for this iteration
            current_batch_size = min(batch_size, remaining)

            prompt = create_prompt(sarcasm, sentiment, dialect, current_batch_size)

            tweets = generate_tweet_batch(prompt, chat)

            # Process the tweets from this batch
            batch_valid_count = 0
            for tweet in tweets:
                if tweet and not tweet.startswith("Error generating tweet"):
                    # Clean the tweet text if needed
                    cleaned_tweet = clean_text(tweet) if 'clean_text' in globals() else tweet

                    # Add to the list
                    all_tweets.append({
                        "text": cleaned_tweet,
                        "sarcasm": str(sarcasm),
                        "sentiment": sentiment,
                        "dialect": dialect,
                        "combined_label": combined_label
                    })
                    batch_valid_count += 1
                    collected_tweets += 1

                    # Stop if we've reached the target number
                    if collected_tweets >= target_number:
                        break

            # If we didn't get any valid tweets in this batch, we might be stuck
            if batch_valid_count == 0:
                print("Warning: No valid tweets generated in this batch. Moving to next configuration.")
                break

            # Add a delay to avoid rate limiting
            if collected_tweets < target_number:  # Only delay if we need more tweets
                time.sleep(2)

    tweets_df = pd.DataFrame(all_tweets)

    csv_path = '/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_simple_prompt/generated_simple_prompt_gemini.csv'
    tweets_df.to_csv(csv_path, index=False, encoding='utf-8')

    print(f"Successfully saved tweets to {csv_path}")

    return tweets_df


In [ ]:
all_tweets = generate_and_save_tweets()

Successfully saved tweets to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_simple_prompt/generated_simple_prompt_gemini.csv


In [ ]:
generated = pd.read_csv('/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_simple_prompt/generated_simple_prompt_gemini.csv')
generated['combined_label'] = generated['sarcasm'].astype(str) + ',' + generated['sentiment'].astype(str) + ',' + generated['dialect'].astype(str)
generated['text'] = generated['text'].apply(clean_text)
generated.to_csv('/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_simple_prompt/generated_simple_prompt_gemini.csv', index=False)

In [ ]:
print_df_info('generated', generated)

*****
len(generated) = 23146

   sarcasm  count
0    False   9623
1     True  13523

  sentiment  count
0       NEG   9411
1       NEU  10398
2       POS   3337
  dialect  count
0   egypt   3601
1    gulf   5617
2  levant   5425
3  magreb   5959
4     msa   2544

      combined_label  count
4    False,NEU,egypt    335
0    False,NEG,egypt    514
8    False,POS,egypt    536
12    True,NEG,egypt    760
6   False,NEU,levant    828
5     False,NEU,gulf    835
2   False,NEG,levant    848
10  False,POS,levant    869
9     False,POS,gulf    938
1     False,NEG,gulf    947
3   False,NEG,magreb    989
7   False,NEU,magreb    990
11  False,POS,magreb    994
16      True,NEG,msa   1074
14   True,NEG,levant   1388
13     True,NEG,gulf   1404
17    True,NEU,egypt   1456
21      True,NEU,msa   1470
15   True,NEG,magreb   1487
19   True,NEU,levant   1492
18     True,NEU,gulf   1493
20   True,NEU,magreb   1499

*****


In [ ]:
import time
def shutdown_sys():
  print('Will Shutdown')
  time.sleep(30)
  from google.colab import runtime
  runtime.unassign()

In [ ]:
shutdown_sys()

Will Shutdown
